Почему важен NLP

Natural Langauge PRocessing

#### Задачи
ML модель - это черный ящик, который на взод получает входящий сигнал X, на вызходе 
Внутри какая-то формула (линейная комбинация - линейными моделями, набор правил как в дереве решений, или вложенных преобразований как в глубокиз нейронных сетях)
Сигнал - вектор
Оцифровать любую информацию и попдавать как веткор
В том числе текст

Какие есть типы данные
- скалярные
  непрерываные
  категориальрные факторы
  ординальные
- векторные
- последовательности (переменной длины)
  текст, аудио, ввидео, движения

Есть примеры моделей, которые умеют работать с разными типами (наример, дерево), но большинство настроено на работу с числовыми данными
Но есть преобразования:
    - категории в число
    - число в категории

Особняком стоят последовательности
Мапинг последовательности в векторное представление = векторизация

Какие практические вопросы нужно для этого решить
- Как разбивать на составные куски<br>большие куски - слишком большая вариативность
- Как мапить куски в числа<br>
- Как агрегировать из в одноу число или вектор<br>

Рассмотрим доступные преобразования текстовых данных. 

Разбиение на составные части = токенизация. Токен - намименьшая составная часть текста
Токен - слово или словосчетание. Но в реальности токены нрезаются динамически, это больше похоже на слог

Предобработка {экстракция, чистка, токенизация, стеминг / лемматизация, }

Мапингов два типа:
- sparse encoding<br>
- dense encoding<br>

Encoding = более наукоемкий термин для векторизации. Из оригниальных объектов 
Decoding = обратный процесс, из веткоров возвращаемся в простроанство оригинальных объектов. См например, генеративные модели

### Токенизация
Цель: найти K подстрок, покрывающих корпус с минимальным числом токенов

Алгоритмы и инструменты:
- BPE (1994)<br>алгоритм компрессии
- BBPE (2019 для GPT-2)
- Tiktoken (реализация BBPE для GPT-3 и далее, 2022)
- WordPiece (2012 для японского языка / 2019 появился в BERT)<br>https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/37842.pdf
- Unigram (2018)
- SentencePiece (2018, сейчас в LLAMA)
- Moses tokenizer (Koehn et al., 2007)<br>чистый набор правил написанный лингвистами, написан для задач MT
- spaCy (2016)<br>принцип работы: подгружаем веса, применяем дефолтный пайплайн к тексту, он выплевывает кучу лексической информации

Детокенизация ("".join) оказывается нетривиальная задача, нужно расставить пробелы

Если хочется обучать с нуля, есть примеры:
- Hugginface
    - BPETrainer
    - WordPieceTrainer
- SentencePiece

Почему может понадобиться<br>
Rust et al., 2021 — "How Good is Your Tokenizer?: токенизаторы, обученные на несовпадающем с задачей домене, существенно роняют F1. Это значит, что важен не только алгоритм, но и на каких данных обучен словарь

Если хочется использовать готовые, есть примеры<br>
- Hugginface
    - AutoTokenizer BPE
    - AutoTokenizer SentencePiece
- Tiktoken
- spaCy
- Moses

Для обучения достаточно 1-10GB текста. Для редких языков нужно больше текста. На что обычно смотрят:
- покрытие
- частотность би-грамм
- доменное соотвествие
- метрика Fertility: кол-во токенов на слово<br>https://medium.com/@biswanai92/understanding-token-fertility-why-it-matters-for-multilingual-llms-38c0b9f20da2

### Sparse
После нарезки на токены, мы составляем словарь всех встретившихся токенов
Будем рассматривать такое описательное пространство, у которого одно измерение = один токен из словаря

Если токен встречается в тексте, в соотвествующем элементе вектора проставляется единичка (факт наличия), либо кол-во раз, сколько оно встречается

Таким образом, токен - это вектор с кучей нулей и одним ненулевым элементом. А каждый текст - это вектор с кучеф нулей некоторый Frequency вектор

Поэтому он называется Sparse представлением

Bag-of-words - модель текста, не учитывающая порядок

OOV - проблема, когда приходит новый токен, не встречавшийся ранее

Плюсы:
- не нужна агрегация, она тут возникает естесвтенно
- все очень интерпретируемо

Минусы:
- не учитывается порядок слов
- конекстно независмы

### TF-IDF

ToDO
- статистика по языкам
- 

**Задача.** Дан корпус `D` (последовательность слов/текстов) и ограничение на размер словаря `|V| ≤ K`. Нужно найти сегментацию, которая оптимально представляет данные.

**Разные алгоритмы формализуют "оптимально" по-разному:**

**BPE — эвристическая задача сжатия.** На каждом шаге ищем пару `(a, b)`, для которой:

$$\text{merge}^* = \arg\max_{(a,b)} \text{count}(a, b, \mathcal{D})$$

то есть жадно максимизируем число сохранённых слияний (≈ минимизируем длину корпуса в токенах). Это жадная аппроксимация задачи минимального описания.

**WordPiece — максимизация правдоподобия унигрэмной модели с добавлением пары.** Слияние выбирается как:

$$\text{merge}^* = \arg\max_{(a,b)} \frac{P(ab)}{P(a) \cdot P(b)}$$

**Unigram LM — строгая вероятностная постановка.** Пусть `V` — текущий словарь. Определим унигрэмную языковую модель с параметрами `p(x)` для каждого токена `x ∈ V`. Токенизация слова `w` — это:

$$x^* = \arg\max_{x \in \text{Seg}(w)} \sum_i \log p(x_i)$$

Обучение — это максимизация суммарного лог-правдоподобия корпуса:

$$\mathcal{L}(V, p) = \sum_{w \in \mathcal{D}} \log P(w) = \sum_{w \in \mathcal{D}} \log \sum_{x \in \text{Seg}(w)} \prod_i p(x_i)$$

при ограничении `|V| ≤ K`. Параметры `p` оцениваются EM-алгоритмом, а словарь урезается по критерию наименьшего влияния на `L`.

**Общая формулировка (MDL-перспектива).** Все subword-алгоритмы можно рассматривать как приближённое решение задачи **Minimum Description Length**:

$$V^*, s^* = \arg\min_{V, s} \; |V| \cdot \text{cost\_per\_type} + \sum_{w \in \mathcal{D}} |s(w)|$$

где `s(w)` — токенизация слова `w`, `|s(w)|` — её длина в токенах. Иными словами: минимизируем совместную стоимость (словарь + длина корпуса в токенах). При ограничении `|V| = K` это превращается в: "найти K подстрок, покрывающих корпус с минимальным числом токенов".

